# Extract data from ERA5-LAND path

## inquiry by year

In [ ]:
# era5land_loader.py
# -*- coding: utf-8 -*-
"""
输入：地名、年份、ERA5-Land 根目录、城市经纬字典
输出：形状为 (365, X, 20, 20, 24) 的 numpy.float32
特性：
- 自动发现 feature 目录（最多 50），自动推断变量名
- 兼容 (time, step) → 展平成 1D 小时轴并去重
- 经纬度支持 0..360 / -180..180；以最近格点为中心取 20×20
- 首次查询缓存 {year}_{city}.pth（pickle），后续命中直接返回
依赖：xarray, cfgrib, eccodes, numpy
"""

import os
import re
import pickle
from typing import List, Tuple, Dict, Optional

import numpy as np
import xarray as xr


# ==========================
# 工具 & 兼容性函数
# ==========================
def _clip_to_year(ds: xr.Dataset, year: int) -> xr.Dataset:
    """仅保留该年的小时：YYYY-01-01 00:00 ~ YYYY-12-31 23:00（含端点）"""
    start = np.datetime64(f"{year}-01-01T00:00:00")
    end   = np.datetime64(f"{year}-12-31T23:00:00")
    if "time" not in ds.coords:
        return ds
    return ds.sel(time=slice(start, end))


def _safe_city_key(city: str) -> str:
    return re.sub(r'[^A-Za-z0-9_]+', '_', city.strip().lower())


def get_latlon(city: str, city_geo_dic: Dict[str, Tuple[float, float]]) -> Tuple[float, float]:
    if city not in city_geo_dic:
        raise KeyError(f"city '{city}' not found in city_geo_dic.")
    lat, lon = city_geo_dic[city]
    return float(lat), float(lon)


def _get_lat_lon_names(ds: xr.Dataset) -> Tuple[str, str]:
    lat_candidates = ["latitude", "lat", "Latitude", "LAT"]
    lon_candidates = ["longitude", "lon", "Longitude", "LON"]
    lat_name = next((c for c in lat_candidates if c in ds.coords), None)
    lon_name = next((c for c in lon_candidates if c in ds.coords), None)
    if lat_name is None or lon_name is None:
        raise KeyError(f"Cannot find latitude/longitude coords in dataset. coords={list(ds.coords)}")
    return lat_name, lon_name


def to_dataset_lon(lon_deg: float, lon_coords: np.ndarray) -> float:
    lon_min, lon_max = float(np.min(lon_coords)), float(np.max(lon_coords))
    if lon_min >= 0 and lon_max > 180:  # 0..360
        return (lon_deg + 360.0) % 360.0
    return ((lon_deg + 180.0) % 360.0) - 180.0  # -180..180


# ==========================
# 目录扫描 & 变量名推断
# ==========================
def list_available_features(data_root: str, year: int, max_features: int = 50) -> List[str]:
    feats = []
    if not os.path.isdir(data_root):
        raise FileNotFoundError(f"data_root not found: {data_root}")
    for name in sorted(os.listdir(data_root)):
        dir1 = os.path.join(data_root, name)
        if not os.path.isdir(dir1):
            continue
        dir_year = os.path.join(dir1, str(year))
        if not os.path.isdir(dir_year):
            continue
        if any(fn.endswith(".grib") for fn in os.listdir(dir_year)):
            feats.append(name)
    if not feats:
        raise FileNotFoundError(f"No feature folders with GRIB files found for year={year} under {data_root}")
    return feats[:max_features]


def infer_var_name(ds: xr.Dataset, feature_folder_name: str) -> str:
    # 1) exact/包含匹配
    for v in ds.data_vars:
        if v == feature_folder_name or (v in feature_folder_name) or (feature_folder_name in v):
            return v
    # 2) 常见别名
    alias = {
        "2m_dewpoint_temperature": "d2m",
        "2m_temperature": "t2m",
        "total_precipitation": "tp",
        "surface_pressure": "sp",
        "u_component_of_wind_10m": "u10",
        "v_component_of_wind_10m": "v10",
    }
    for k, v in alias.items():
        if k in feature_folder_name and v in ds.data_vars:
            return v
    # 3) fallback
    return list(ds.data_vars)[0]


# ==========================
# (time, step) 展平 & 去重
# ==========================
def _flatten_time_step(ds: xr.Dataset) -> xr.Dataset:
    """
    将 (time, step) 合成 1D 时间轴：
      - 计算 time_flat = time[:,None] + step[None,:]
      - stack 成单维 'ts'
      - drop_vars 删除旧的 'time'/'step' 坐标（避免命名冲突）
      - 用新坐标替换维度 → 'time'，排序并去重（保留同一时刻的最后一次）
    若无 step，原样返回；若有 'number' 集合维，取第一个成员。
    """
    if "number" in ds.dims:
        ds = ds.isel(number=0, drop=True)
    if "step" not in ds.dims:
        return ds

    t = ds["time"].values  # (Nt,)
    s = ds["step"].values  # (Ns,)
    time_flat = (t[:, None] + s[None, :]).reshape(-1)

    ds = ds.stack(ts=("time", "step"))
    ds = ds.drop_vars([name for name in ("time", "step") if name in ds.coords], errors="ignore")
    ds = ds.assign_coords(valid_time=("ts", time_flat))
    ds = ds.swap_dims({"ts": "valid_time"}).rename({"valid_time": "time"}).sortby("time")

    # 去重：保留同一时刻的最后一次（一般是较大 step）
    vals = ds["time"].values
    _, idx_rev = np.unique(vals[::-1], return_index=True)
    keep = np.sort(vals.size - 1 - idx_rev)
    ds = ds.isel(time=keep)
    return ds


def _drop_duplicate_times(ds: xr.Dataset) -> xr.Dataset:
    if "time" not in ds.coords:
        return ds
    vals = ds["time"].values
    _, idx_rev = np.unique(vals[::-1], return_index=True)
    keep = np.sort(vals.size - 1 - idx_rev)
    return ds.isel(time=keep)


# ==========================
# 数据打开 & 空间索引
# ==========================
def open_feature_year(data_root: str, feature: str, year: int) -> xr.Dataset:
    """
    打开某个 feature 的全年 GRIB 并按 time 维拼接；自动展平 (time, step)。
    期望路径：era5land/<feature>/<year>/reanalysis-era5-land_<feature>_<year>-MM.grib
    若文件名不完全一致，使用包含 year-MM 的 .grib 作为兜底。
    """
    dir_year = os.path.join(data_root, feature, str(year))
    if not os.path.isdir(dir_year):
        raise FileNotFoundError(f"dir not found: {dir_year}")

    files: List[str] = []
    for m in range(1, 13):
        fname = f"reanalysis-era5-land_{feature}_{year}-{m:02d}.grib"
        fpath = os.path.join(dir_year, fname)
        if os.path.exists(fpath):
            files.append(fpath)
        else:
            cand = [fn for fn in os.listdir(dir_year) if fn.endswith(".grib") and f"{year}-{m:02d}" in fn]
            files += [os.path.join(dir_year, fn) for fn in sorted(cand)]
    if not files:
        raise FileNotFoundError(f"No monthly grib files under {dir_year}")

    dsets = []
    for fp in files:
        try:
            ds = xr.open_dataset(
                fp,
                engine="cfgrib",
                backend_kwargs={"indexpath": ""},
                decode_timedelta=True,
            )
        except Exception as e:
            raise RuntimeError(
                f"Failed to open {fp} with engine='cfgrib'. "
                f"Ensure cfgrib + eccodes are installed. Original error: {e}"
            )
        ds = _flatten_time_step(ds)
        dsets.append(ds)

    ds_all = xr.concat(dsets, dim="time")
    ds_all = ds_all.sortby("time")
    ds_all = _drop_duplicate_times(ds_all)
    ds_all = _clip_to_year(ds_all, year)     # ← 新增：裁到目标年份
    return ds_all


def find_20x20_slices(ds: xr.Dataset, lat_c: float, lon_c: float) -> Tuple[slice, slice, Dict]:
    lat_name, lon_name = _get_lat_lon_names(ds)
    lat_arr = ds[lat_name].values
    lon_arr = ds[lon_name].values

    lon_c_ds = to_dataset_lon(lon_c, lon_arr)
    ilat_c = int(np.argmin(np.abs(lat_arr - lat_c)))
    diff_lon = np.abs(((lon_arr - lon_c_ds + 180.0) % 360.0) - 180.0)
    ilon_c = int(np.argmin(diff_lon))

    half = 10
    ilat0, ilat1 = max(0, ilat_c - half), min(len(lat_arr), ilat_c + half)
    ilon0, ilon1 = max(0, ilon_c - half), min(len(lon_arr), ilon_c + half)

    need_lat = 20 - (ilat1 - ilat0)
    if need_lat > 0:
        ilat0 = max(0, ilat0 - need_lat)
    need_lon = 20 - (ilon1 - ilon0)
    if need_lon > 0:
        ilon0 = max(0, ilon0 - need_lon)

    ilat1 = min(len(lat_arr), ilat0 + 20)
    ilon1 = min(len(lon_arr), ilon0 + 20)

    meta = dict(
        lat_center=float(lat_arr[ilat_c]),
        lon_center=float(lon_arr[ilon_c]),
        lat_bounds=(float(lat_arr[min(ilat0, ilat1 - 1)]), float(lat_arr[max(ilat0, ilat1 - 1)])),
        lon_bounds=(float(lon_arr[ilon0]), float(lon_arr[ilon1 - 1])),
        ilat=(int(ilat0), int(ilat1)),
        ilon=(int(ilon0), int(ilon1)),
        lat_name=lat_name,
        lon_name=lon_name,
    )
    return slice(ilat0, ilat1), slice(ilon0, ilon1), meta


# ==========================
# 形状规范化：到 (365, 20, 20, 24)
# ==========================
def feature_to_year_tensor20(ds_all: xr.Dataset,
                             var_name: str,
                             lat_slice: slice,
                             lon_slice: slice) -> np.ndarray:
    lat_name, lon_name = _get_lat_lon_names(ds_all)
    da = ds_all[var_name]  # (time, lat, lon)

    da20 = da.isel({lat_name: lat_slice, lon_name: lon_slice})

    time = da20["time"]
    if hasattr(time, "dt"):
        is_feb29 = (time.dt.month == 2) & (time.dt.day == 29)
        if bool(is_feb29.any()):
            da20 = da20.sel(time=~is_feb29)

    T = da20.sizes["time"]
    if T % 24 != 0:
        raise ValueError(
            f"time length {T} not divisible by 24 after Feb-29 removal. "
            f"Likely not hourly or 'step' not flattened. "
            f"Inspect ds.dims and ds['time'] for details."
        )
    days = T // 24
    if days != 365:
        raise ValueError(f"expected 365 days, got {days}. Check input files/year completeness.")

    arr = da20.values.reshape(days, 24, 20, 20)  # (365, 24, 20, 20)
    arr = np.moveaxis(arr, 1, -1)                # (365, 20, 20, 24)
    return arr.astype("float32")


# ==========================
# 主入口：城市+年份 → (365, X, 20, 20, 24)
# 带 .pth 缓存（year_city.pth）
# ==========================
def load_era5land_city_year(
    city: str,
    year: int,
    data_root: str,
    city_geo_dic: Dict[str, Tuple[float, float]],
    feature_whitelist: Optional[List[str]] = None,
    feature_blacklist: Optional[List[str]] = None,
    cache_dir: str = "era5_cache",
    force_refresh: bool = False,
    max_features: int = 50,
) -> np.ndarray:
    os.makedirs(cache_dir, exist_ok=True)
    cache_name = f"{int(year)}_{_safe_city_key(city)}.pth"
    cache_path = os.path.join(cache_dir, cache_name)
    if (not force_refresh) and os.path.isfile(cache_path):
        with open(cache_path, "rb") as f:
            obj = pickle.load(f)
        arr = obj["arr"]
        if (arr.ndim == 5 and arr.shape[0] == 365 and
                arr.shape[2] == 20 and arr.shape[3] == 20 and arr.shape[4] == 24):
            return arr
        # 缓存不兼容则重建

    lat, lon = get_latlon(city, city_geo_dic)

    features = list_available_features(data_root, year, max_features=max_features)
    if feature_whitelist:
        wl = set(feature_whitelist)
        features = [f for f in features if f in wl]
    if feature_blacklist:
        bl = set(feature_blacklist)
        features = [f for f in features if f not in bl]
    if not features:
        raise RuntimeError("No features left after applying white/black list filters.")

    probe_ds = open_feature_year(data_root, features[0], year)
    lat_slice, lon_slice, region_meta = find_20x20_slices(probe_ds, lat, lon)

    tensors: List[np.ndarray] = []
    for feat in features:
        ds_all = open_feature_year(data_root, feat, year)
        var_name = infer_var_name(ds_all, feat)
        tens = feature_to_year_tensor20(ds_all, var_name, lat_slice, lon_slice)
        tensors.append(tens.astype("float32"))

    arr = np.stack(tensors, axis=1).astype("float32")  # (365, X, 20, 20, 24)

    to_save = {"arr": arr, "city": city, "year": int(year), "features": features, "region_meta": region_meta}
    with open(cache_path, "wb") as f:
        pickle.dump(to_save, f)
    return arr


# ==========================
# 示例（可删）
# ==========================
if __name__ == "__main__":
    city_geo_dic = {
        "Leeds": (53.7974185, -1.5437941),
        "Beijing": (39.9042, 116.4074),
    }
    data_root = "era5land"

    try:
        arr = load_era5land_city_year(
            city="Leeds",
            year=1997,
            data_root=data_root,
            city_geo_dic=city_geo_dic,
            cache_dir="era5_cache",
            force_refresh=False,
            max_features=50,
        )
        print("Loaded array shape:", arr.shape)  # (365, X, 20, 20, 24)
    except Exception as e:
        print("Error:", e)

## inqury by day

In [ ]:
# era5land_loader.py
# -*- coding: utf-8 -*-
"""
输入：地名、年份、ERA5-Land 根目录、城市经纬字典
输出：形状为 (365, X, 20, 20, 24) 的 numpy.float32
特性：
- 自动发现 feature 目录（最多 50），自动推断变量名
- 兼容 (time, step) → 展平成 1D 小时轴并去重
- 经纬度支持 0..360 / -180..180；以最近格点为中心取 20×20
- 首次查询缓存 {year}_{city}.pth（pickle），后续命中直接返回
依赖：xarray, cfgrib, eccodes, numpy
"""

import os
import re
import pickle
from typing import List, Tuple, Dict, Optional

import numpy as np
import xarray as xr


# ==========================
# 工具 & 兼容性函数
# ==========================
def _clip_to_year(ds: xr.Dataset, year: int) -> xr.Dataset:
    """仅保留该年的小时：YYYY-01-01 00:00 ~ YYYY-12-31 23:00（含端点）"""
    start = np.datetime64(f"{year}-01-01T00:00:00")
    end   = np.datetime64(f"{year}-12-31T23:00:00")
    if "time" not in ds.coords:
        return ds
    return ds.sel(time=slice(start, end))


def _safe_city_key(city: str) -> str:
    return re.sub(r'[^A-Za-z0-9_]+', '_', city.strip().lower())


def get_latlon(city: str, city_geo_dic: Dict[str, Tuple[float, float]]) -> Tuple[float, float]:
    if city not in city_geo_dic:
        raise KeyError(f"city '{city}' not found in city_geo_dic.")
    lat, lon = city_geo_dic[city]
    return float(lat), float(lon)


def _get_lat_lon_names(ds: xr.Dataset) -> Tuple[str, str]:
    lat_candidates = ["latitude", "lat", "Latitude", "LAT"]
    lon_candidates = ["longitude", "lon", "Longitude", "LON"]
    lat_name = next((c for c in lat_candidates if c in ds.coords), None)
    lon_name = next((c for c in lon_candidates if c in ds.coords), None)
    if lat_name is None or lon_name is None:
        raise KeyError(f"Cannot find latitude/longitude coords in dataset. coords={list(ds.coords)}")
    return lat_name, lon_name


def to_dataset_lon(lon_deg: float, lon_coords: np.ndarray) -> float:
    lon_min, lon_max = float(np.min(lon_coords)), float(np.max(lon_coords))
    if lon_min >= 0 and lon_max > 180:  # 0..360
        return (lon_deg + 360.0) % 360.0
    return ((lon_deg + 180.0) % 360.0) - 180.0  # -180..180


# ==========================
# 目录扫描 & 变量名推断
# ==========================
def list_available_features(data_root: str, year: int, max_features: int = 50) -> List[str]:
    feats = []
    if not os.path.isdir(data_root):
        raise FileNotFoundError(f"data_root not found: {data_root}")
    for name in sorted(os.listdir(data_root)):
        dir1 = os.path.join(data_root, name)
        if not os.path.isdir(dir1):
            continue
        dir_year = os.path.join(dir1, str(year))
        if not os.path.isdir(dir_year):
            continue
        if any(fn.endswith(".grib") for fn in os.listdir(dir_year)):
            feats.append(name)
    if not feats:
        raise FileNotFoundError(f"No feature folders with GRIB files found for year={year} under {data_root}")
    return feats[:max_features]


def infer_var_name(ds: xr.Dataset, feature_folder_name: str) -> str:
    # 1) exact/包含匹配
    for v in ds.data_vars:
        if v == feature_folder_name or (v in feature_folder_name) or (feature_folder_name in v):
            return v
    # 2) 常见别名
    '''
    alias = {
        "2m_dewpoint_temperature": "d2m",
        "2m_temperature": "t2m",
        "total_precipitation": "tp",
        "surface_pressure": "sp",
        "u_component_of_wind_10m": "u10",
        "v_component_of_wind_10m": "v10",
    }
    for k, v in alias.items():
        if k in feature_folder_name and v in ds.data_vars:
            return v
    '''
    # 3) fallback
    return list(ds.data_vars)[0]


# ==========================
# (time, step) 展平 & 去重
# ==========================
def _flatten_time_step(ds: xr.Dataset) -> xr.Dataset:
    """
    将 (time, step) 合成 1D 时间轴（小时级）并去重：
      1) 若有集合维 'number'，先取第一个成员
      2) 若无 'step' 维，原样返回
      3) 计算扁平有效时刻 time_flat = time[:,None] + step[None,:]
      4) stack 成单维 'ts'，然后直接用 assign_coords 覆盖 'ts' 坐标为 time_flat
      5) 将维度/坐标 'ts' 重命名为 'time'，排序并去重（保留同一时刻的最后一次）
    """
    if "number" in ds.dims:
        ds = ds.isel(number=0, drop=True)
    if "step" not in ds.dims:
        return ds

    # 计算扁平有效时间轴
    t = ds["time"].values  # (Nt,)
    s = ds["step"].values  # (Ns,)
    time_flat = (t[:, None] + s[None, :]).reshape(-1)

    # 把 (time, step) 叠成一维 'ts'
    ds = ds.stack(ts=("time", "step"))

    # 关键：不要删除 MultiIndex 的层级；直接“覆盖” ts 坐标为扁平时间
    ds = ds.assign_coords(ts=("ts", time_flat))

    # 此时 'ts' 已不是 MultiIndex，直接改名为 'time' 并排序
    ds = ds.rename({"ts": "time"}).sortby("time")

    # 去重：保留同一时刻的最后一次（一般对应较大的 step）
    vals = ds["time"].values
    _, idx_rev = np.unique(vals[::-1], return_index=True)
    keep = np.sort(vals.size - 1 - idx_rev)
    ds = ds.isel(time=keep)

    return ds


def _drop_duplicate_times(ds: xr.Dataset) -> xr.Dataset:
    if "time" not in ds.coords:
        return ds
    vals = ds["time"].values
    _, idx_rev = np.unique(vals[::-1], return_index=True)
    keep = np.sort(vals.size - 1 - idx_rev)
    return ds.isel(time=keep)


# ==========================
# 数据打开 & 空间索引
# ==========================
def open_feature_year(data_root: str, feature: str, year: int) -> xr.Dataset:
    """
    打开某个 feature 的全年 GRIB 并按 time 维拼接；自动展平 (time, step)。
    期望路径：era5land/<feature>/<year>/reanalysis-era5-land_<feature>_<year>-MM.grib
    若文件名不完全一致，使用包含 year-MM 的 .grib 作为兜底。
    """
    dir_year = os.path.join(data_root, feature, str(year))
    if not os.path.isdir(dir_year):
        raise FileNotFoundError(f"dir not found: {dir_year}")

    files: List[str] = []
    for m in range(1, 13):
        fname = f"reanalysis-era5-land_{feature}_{year}-{m:02d}.grib"
        fpath = os.path.join(dir_year, fname)
        if os.path.exists(fpath):
            files.append(fpath)
        else:
            cand = [fn for fn in os.listdir(dir_year) if fn.endswith(".grib") and f"{year}-{m:02d}" in fn]
            files += [os.path.join(dir_year, fn) for fn in sorted(cand)]
    if not files:
        raise FileNotFoundError(f"No monthly grib files under {dir_year}")

    dsets = []
    for fp in files:
        try:
            ds = xr.open_dataset(
                fp,
                engine="cfgrib",
                backend_kwargs={"indexpath": ""},
                decode_timedelta=True,
            )
        except Exception as e:
            raise RuntimeError(
                f"Failed to open {fp} with engine='cfgrib'. "
                f"Ensure cfgrib + eccodes are installed. Original error: {e}"
            )
        ds = _flatten_time_step(ds)
        dsets.append(ds)

    ds_all = xr.concat(dsets, dim="time")
    ds_all = ds_all.sortby("time")
    ds_all = _drop_duplicate_times(ds_all)
    ds_all = _clip_to_year(ds_all, year)     # ← 新增：裁到目标年份
    return ds_all


def find_20x20_slices(ds: xr.Dataset, lat_c: float, lon_c: float) -> Tuple[slice, slice, Dict]:
    lat_name, lon_name = _get_lat_lon_names(ds)
    lat_arr = ds[lat_name].values
    lon_arr = ds[lon_name].values

    lon_c_ds = to_dataset_lon(lon_c, lon_arr)
    ilat_c = int(np.argmin(np.abs(lat_arr - lat_c)))
    diff_lon = np.abs(((lon_arr - lon_c_ds + 180.0) % 360.0) - 180.0)
    ilon_c = int(np.argmin(diff_lon))

    half = 10
    ilat0, ilat1 = max(0, ilat_c - half), min(len(lat_arr), ilat_c + half)
    ilon0, ilon1 = max(0, ilon_c - half), min(len(lon_arr), ilon_c + half)

    need_lat = 20 - (ilat1 - ilat0)
    if need_lat > 0:
        ilat0 = max(0, ilat0 - need_lat)
    need_lon = 20 - (ilon1 - ilon0)
    if need_lon > 0:
        ilon0 = max(0, ilon0 - need_lon)

    ilat1 = min(len(lat_arr), ilat0 + 20)
    ilon1 = min(len(lon_arr), ilon0 + 20)

    meta = dict(
        lat_center=float(lat_arr[ilat_c]),
        lon_center=float(lon_arr[ilon_c]),
        lat_bounds=(float(lat_arr[min(ilat0, ilat1 - 1)]), float(lat_arr[max(ilat0, ilat1 - 1)])),
        lon_bounds=(float(lon_arr[ilon0]), float(lon_arr[ilon1 - 1])),
        ilat=(int(ilat0), int(ilat1)),
        ilon=(int(ilon0), int(ilon1)),
        lat_name=lat_name,
        lon_name=lon_name,
    )
    return slice(ilat0, ilat1), slice(ilon0, ilon1), meta


# ==========================
# 形状规范化：到 (365, 20, 20, 24)
# ==========================
def feature_to_year_tensor20(ds_all: xr.Dataset,
                             var_name: str,
                             lat_slice: slice,
                             lon_slice: slice) -> np.ndarray:
    lat_name, lon_name = _get_lat_lon_names(ds_all)
    da = ds_all[var_name]  # (time, lat, lon)

    da20 = da.isel({lat_name: lat_slice, lon_name: lon_slice})

    time = da20["time"]
    if hasattr(time, "dt"):
        is_feb29 = (time.dt.month == 2) & (time.dt.day == 29)
        if bool(is_feb29.any()):
            da20 = da20.sel(time=~is_feb29)

    T = da20.sizes["time"]
    if T % 24 != 0:
        raise ValueError(
            f"time length {T} not divisible by 24 after Feb-29 removal. "
            f"Likely not hourly or 'step' not flattened. "
            f"Inspect ds.dims and ds['time'] for details."
        )
    days = T // 24
    if days != 365:
        raise ValueError(f"expected 365 days, got {days}. Check input files/year completeness.")

    arr = da20.values.reshape(days, 24, 20, 20)  # (365, 24, 20, 20)
    arr = np.moveaxis(arr, 1, -1)                # (365, 20, 20, 24)
    return arr.astype("float32")


# ==========================
# 主入口：城市+年份 → (365, X, 20, 20, 24)
# 带 .pth 缓存（year_city.pth）
# ==========================
def load_era5land_city_year(
    city: str,
    year: int,
    data_root: str,
    city_geo_dic: Dict[str, Tuple[float, float]],
    feature_whitelist: Optional[List[str]] = None,
    feature_blacklist: Optional[List[str]] = None,
    cache_dir: str = "era5_cache",
    force_refresh: bool = False,
    max_features: int = 50,
) -> np.ndarray:
    os.makedirs(cache_dir, exist_ok=True)
    cache_name = f"{int(year)}_{_safe_city_key(city)}.pth"
    cache_path = os.path.join(cache_dir, cache_name)
    if (not force_refresh) and os.path.isfile(cache_path):
        with open(cache_path, "rb") as f:
            obj = pickle.load(f)
        arr = obj["arr"]
        if (arr.ndim == 5 and arr.shape[0] == 365 and
                arr.shape[2] == 20 and arr.shape[3] == 20 and arr.shape[4] == 24):
            return arr
        # 缓存不兼容则重建

    lat, lon = get_latlon(city, city_geo_dic)

    features = list_available_features(data_root, year, max_features=max_features)
    if feature_whitelist:
        wl = set(feature_whitelist)
        features = [f for f in features if f in wl]
    if feature_blacklist:
        bl = set(feature_blacklist)
        features = [f for f in features if f not in bl]
    if not features:
        raise RuntimeError("No features left after applying white/black list filters.")

    probe_ds = open_feature_year(data_root, features[0], year)
    lat_slice, lon_slice, region_meta = find_20x20_slices(probe_ds, lat, lon)

    tensors: List[np.ndarray] = []
    for feat in features:
        ds_all = open_feature_year(data_root, feat, year)
        var_name = infer_var_name(ds_all, feat)
        tens = feature_to_year_tensor20(ds_all, var_name, lat_slice, lon_slice)
        tensors.append(tens.astype("float32"))

    arr = np.stack(tensors, axis=1).astype("float32")  # (365, X, 20, 20, 24)

    to_save = {"arr": arr, "city": city, "year": int(year), "features": features, "region_meta": region_meta}
    with open(cache_path, "wb") as f:
        pickle.dump(to_save, f)
    return arr


# ==========================
# 示例（可删）
# ==========================
if __name__ == "__main__":
    city_geo_dic={
        'Leeds': (53.7974185, -1.5437941),
        'Bristol': (51.4538022, -2.5972985),
        'Newcastle': (54.9738474, -1.6131572),
        'Nottingham': (52.9534193, -1.1496461),
        'Liverpool': (53.4071991, -2.99168),
        'Sheffield': (53.3806626, -1.4702278),
        'Reading': (51.4514953, -0.9836342),
        'Bury': (52.2460367, 0.7125173),
        'Hounslow': (51.4686132, -0.3613471),
        'Croydon': (51.3713049, -0.101957),
        'Birmingham': (52.4796992, -1.9026911),
        'Middlesborough': (51.6107383, -0.0610164),
        'Stoke': (53.0162014, -2.1812607),
        'Glasgow': (55.861155, -4.2501687),
        'Cardiff': (51.4816546, -3.1791934),
        'Edinburgh': (55.9533456, -3.1883749),
        'Oxford': (51.7520131, -1.2578499),
        'Manchester': (53.4794892, -2.2451148),
        'Barts': (51.5175315, -0.0998302),
        'Swansea': (51.6195955, -3.9459248),
        'Wrexham': (53.0465084, -2.9937869)
    }
    data_root = "era5land"

    try:
        arr = load_era5land_city_year(
            city="Oxford",
            year=1997,
            data_root=data_root,
            city_geo_dic=city_geo_dic,
            cache_dir="era5_cache",
            force_refresh=False,
            max_features=50,
        )
        print("Loaded array shape:", arr.shape)  # (365, X, 20, 20, 24)
    except Exception as e:
        print("Error:", e)